**1. Cargar el archivo entregas_retrasos.csv con pandas y revisar su forma (shape) y las primeras filas.**

1.1. Carga de las librerías necesarias

In [ ]:
import pandas as pdimport seaborn as snsimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreimport numpy as np

1.2. Carga del archivo y revisión del shape

In [ ]:
# Sube el archivo entregas_retrasos.csv a Colab (icono de carpeta a la izquierda) antes de correr esta celdadf = pd.read_csv('entregas_retrasos.csv')print("Estructura y datos del dataset (filas, columnas):", df.shape)display(df.head())

**2. Revisar los tipos de dato de cada columna y verificar si existen valores nulos.**

2.1 Revisión de cada tipo de datos de las columnas

In [ ]:
print(("-" * 15), "Información del Dataset", ("-" * 15))df.info()

2.2 Verificación de si existen valores nulos

In [ ]:
print(("-" * 15), "Valores nulos por cada columna", ("-" * 15))print(df.isnull().sum())

**3. Convertir la columna fecha_pedido a tipo datetime y extraer variables nuevas útiles: mes, día de la semana, hora, y si fue en fin de semana.**

3.1 Convertir la columna fecha_pedido a tipo datetime

In [ ]:
df['fecha_pedido'] = pd.to_datetime(df['fecha_pedido'])

3.2 Extracción de las nuevas variables útiles: mes, día de la semana, hora y fin de semana

In [ ]:
df['mes'] = df['fecha_pedido'].dt.monthdf['dia_semana'] = df['fecha_pedido'].dt.dayofweekdf['hora'] = df['fecha_pedido'].dt.hourdf['fin_de_semana'] = (df['fecha_pedido'].dt.dayofweek >= 5).astype(int)

3.3 Comprobación de que se crearan correctamente

In [ ]:
print(("-" * 15), "Nuevas columnas creadas", ("-" * 15))print(df[['fecha_pedido', 'mes', 'dia_semana', 'hora', 'fin_de_semana']].head())

**4. Convertir la columna prioridad (Alta/Media/Baja) a valores numéricos ordinales.**

4.1 Conversión de prioridad a escala numérica

In [ ]:
df['prioridad'] = df['prioridad'].map({'Alta': 2, 'Media': 1, 'Baja': 0})

4.2 Verificación de la conversión

In [ ]:
print(("-" * 15), "Valores únicos en prioridad", ("-" * 15))print(df['prioridad'].unique())

**5. Eliminar columnas que no deben usarse como variables predictoras (el identificador de la entrega y la fecha original ya descompuesta).**

5.1 Eliminar las columnas que no son predictivas

In [ ]:
df = df.drop(columns=['id_entrega', 'fecha_pedido'])

5.2 Verificación de que se quitaran las columnas

In [ ]:
print(("-" * 15), "Columnas actuales del dataset", ("-" * 15))print(df.columns.tolist())

**6. Aplicar codificación one-hot (pd.get_dummies) a las variables categóricas: hospital_ciudad, vacuna y proveedor.**

6.1 Aplicación de la codificación one-hot a las variables categóricas

In [ ]:
df_encoded = pd.get_dummies(df, columns=['hospital_ciudad', 'vacuna', 'proveedor'], drop_first=True)

6.2 Verificación de las nuevas columnas que se generaron

In [ ]:
print(("-" * 15), "Columnas después de One-Hot Encoding", ("-" * 15))print(df_encoded.columns.tolist())

6.3 Verificación de la nueva "shape" del dataset

In [ ]:
print(("-" * 15), "Nueva forma del dataset (filas, columnas)", ("-" * 15))print(df_encoded.shape)

**7. Generar un mapa de calor de correlación entre las variables numéricas para identificar relaciones relevantes con la variable objetivo.**

7.1 Cálculo de la matriz de correlación de todas las variables numéricas

In [ ]:
correlacion = df_encoded.corr()

7.2 Generación del mapa de calor

In [ ]:
plt.figure(figsize=(14, 10))sns.heatmap(correlacion, annot=False, cmap='coolwarm', center=0)plt.title('Mapa de Calor de Correlación - Retraso de Entregas')plt.tight_layout()plt.show()

**8. Definir las variables predictoras (X) y la variable objetivo (y = retraso_dias), separando en conjunto de entrenamiento y de prueba (80/20).**

8.1 Definición de la variable objetivo (y)

In [ ]:
y = df_encoded['retraso_dias']

8.2 Definición de las variables predictoras (X) excluyendo la variable objetivo

In [ ]:
X = df_encoded.drop(columns=['retraso_dias'])

8.3 Dividir en conjunto de entrenamiento (80%) y prueba (20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

8.4 Verificación de las dimensiones de los conjuntos

In [ ]:
print("Forma de X_train:", X_train.shape)print("Forma de X_test:", X_test.shape)print("Forma de y_train:", y_train.shape)print("Forma de y_test:", y_test.shape)

**9. Entrenar un modelo de regresión (RandomForestRegressor) con los datos de entrenamiento.**

9.1 Creación de una instancia de Random Forest

In [ ]:
modelo = RandomForestRegressor(random_state=42)

9.2 Entrenamos el modelo con los datos de entrenamiento

In [ ]:
modelo.fit(X_train, y_train)print("Modelo entrenado con exito...")

**10. Evaluar el modelo con MAE, RMSE y R² sobre entrenamiento y prueba.**

10.1 Predicciones sobre entrenamiento y prueba

In [ ]:
pred_train = modelo.predict(X_train)pred_test = modelo.predict(X_test)

10.2 Cálculo de métricas

In [ ]:
mae_test = mean_absolute_error(y_test, pred_test)rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))r2_train = r2_score(y_train, pred_train)r2_test = r2_score(y_test, pred_test)print("=" * 50)print("MÉTRICAS DEL MODELO - Retraso de Entregas")print("=" * 50)print(f"R2 (entrenamiento): {r2_train:.4f}")print(f"R2 (prueba):        {r2_test:.4f}")print(f"MAE (prueba):       {mae_test:.2f} días")print(f"RMSE (prueba):      {rmse_test:.2f} días")print("=" * 50)

**11. Analizar la importancia de las variables del modelo entrenado.**

In [ ]:
importancias = modelo.feature_importances_df_importancias = pd.DataFrame({'Variable': X.columns, 'Importancia': importancias})df_importancias = df_importancias.sort_values('Importancia', ascending=False).reset_index(drop=True)plt.figure(figsize=(10, 8))plt.barh(df_importancias['Variable'], df_importancias['Importancia'], color='steelblue')plt.gca().invert_yaxis()plt.title('Importancia de las Variables en la Predicción del Retraso de Entregas')plt.xlabel('Importancia (Proporción)')plt.ylabel('Variable')plt.tight_layout()plt.show()print(df_importancias)

**12. Redactar una breve conclusión con los hallazgos principales (predicción de ejemplo con un registro construido manualmente).**

In [ ]:
ejemplo_entrega = {col: 0 for col in X.columns}ejemplo_entrega.update({    'hospital_capacidad_almacen': 4000,    'proveedor_tiempo_entrega_dias': 16,    'proveedor_variabilidad_dias': 4,    'prioridad': 2,    'cantidad_entregada': 250,    'mes': 10,    'dia_semana': 5,    'hora': 14,    'fin_de_semana': 1,})# Activa aquí las columnas one-hot que correspondan a tu ejemplo (ej. 'hospital_ciudad_Ciudad Sur': 1)ejemplo_df = pd.DataFrame([ejemplo_entrega])[X.columns]prediccion_ejemplo = modelo.predict(ejemplo_df)[0]print("=" * 60)print("PREDICCIÓN DE EJEMPLO")print("=" * 60)print(f"Entrega de prioridad Alta, 250 dosis, proveedor con lead time de 16 días.")print(f"Retraso predicho por el modelo: {prediccion_ejemplo:.2f} días.")print("=" * 60)

1. Variables más importantes y lógica de negocioCompleta aquí cuáles variables resultaron más importantes según tu gráfico y por qué tiene sentido de negocio (ej. el tiempo de entrega del proveedor y su variabilidad deberían dominar).2. Diferencia de R2 y sobreajusteCompara el R2 de entrenamiento vs. prueba y concluye si hay sobreajuste.3. Error promedio del modelo (MAE)Interpreta el MAE en días: ¿es un margen de error aceptable para planificación logística?4. Codificación One-Hot y drop_firstExplica por qué se usó drop_first=True al codificar hospital_ciudad, vacuna y proveedor.5. Nueva variable a recolectarPropón una variable adicional que ayudaría a mejorar el modelo (ej. número de fallas de cadena de frío en tránsito, clima en la ruta, etc.).